# LLM Adaptation Workflow
### Notebook 01 — Foundation Models & Transformers

---

## What is a foundation model?

A **foundation model** is a large neural network trained on enormous amounts of text — billions of web pages, books, code, and articles. The training process teaches the model to predict the next word in a sequence, which, at sufficient scale, forces it to learn grammar, facts, reasoning patterns, and even some degree of logic.

Examples you might have heard of:
- **GPT-4** (OpenAI) — closed source, API only
- **Llama 3** (Meta) — open source, can run locally
- **Phi-2 / Phi-3** (Microsoft) — small but surprisingly capable open-source models
- **Mistral** — open-source, strong performance per parameter

We'll use **TinyLlama** (1.1 billion parameters) and **Phi-2** (2.7 billion parameters) because they're small enough to run comfortably on a Mac but demonstrate all the same concepts as their larger cousins.

---

## How do transformers work? (the short version)

All modern LLMs are built on an architecture called the **Transformer**, introduced by Google in 2017.

The key insight is **attention**: instead of reading text left-to-right like older models, a transformer can look at all words simultaneously and learn which words are most relevant to each other.

```
Input text  →  Tokeniser  →  Token IDs  →  Embeddings  →  Attention layers  →  Output

"The stock    ["The",       [1, 234,      [[0.2, -0.1,    (each token         "rose"
 price..."     "stock",      891, ...]     0.8, ...],      attends to
               "price"]                    ...]            all others)
```

A few things to know:
- The model operates on **tokens** (roughly word-pieces), not raw characters
- Each token is converted to an **embedding** — a list of numbers representing its meaning
- **Attention** lets the model weight how much each token should influence every other token
- The model has many **layers** stacked on top of each other, each refining the representation
- At the end, the model outputs a probability distribution over the vocabulary — it's always predicting the next token

---

## What are parameters?

A model's **parameters** (or **weights**) are the numbers that store everything the model has learned. TinyLlama has 1.1 billion of them. GPT-4 has an estimated 1.7 trillion.

When we fine-tune, we adjust these numbers. When we use LoRA (covered in Notebook 04), we adjust only a tiny fraction of them — which is what makes fine-tuning feasible on a laptop.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
# transformers: Hugging Face's library for loading and running pre-trained models
# torch: PyTorch — the deep learning framework everything is built on

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Detect device (MPS = Apple Silicon, cuda = NVIDIA, cpu = fallback)
device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

---

## 1. Loading a model from Hugging Face

Hugging Face's **Model Hub** hosts thousands of open-source models. We can load any of them with two lines of code.

We load two things:
- **Tokeniser** — converts text to token IDs and back
- **Model** — the actual neural network

First time you run this, it downloads the model (~600 MB for TinyLlama). After that it's cached locally.

In [ ]:
# We'll use TinyLlama — a 1.1B parameter model that's fast to load and run.
# It was trained on 3 trillion tokens of text, same architecture as Llama 2.
# Model card: https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokeniser...")
tokeniser = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # Use 16-bit floats to halve memory usage
    device_map="auto",            # Automatically place layers on available device
)

print("\nModel loaded successfully!")
print(f"Model type   : {type(model).__name__}")

---

## 2. Understanding the tokeniser

Before text reaches the model, it must be converted to numbers. The tokeniser handles this.

Tokens are not always whole words — they're sub-word pieces. For example, "financial" might be split into "fin" + "ancial". This allows the model to handle rare words by breaking them into known sub-pieces.

In [ ]:
# Let's see how the tokeniser breaks down a financial sentence

example_text = "The company's revenue grew 14% year-over-year driven by strong EPS."

# Encode: text → token IDs
token_ids = tokeniser.encode(example_text)

# Convert IDs back to human-readable tokens (not full words — sub-word pieces)
tokens = tokeniser.convert_ids_to_tokens(token_ids)

print("Original text:")
print(f"  {example_text}\n")

print("Tokens (sub-word pieces):")
print(f"  {tokens}\n")

print("Token IDs (what the model actually sees):")
print(f"  {token_ids}\n")

print(f"Character count : {len(example_text)}")
print(f"Token count     : {len(token_ids)}")
print(f"\nNote: models have a 'context window' — a maximum number of tokens they can process at once.")
print(f"TinyLlama's context window: 2048 tokens")

---

## 3. Inspecting the model's architecture

Let's look inside the model to understand its structure. This is useful for understanding where LoRA adapters will be inserted later.

In [ ]:
# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters    : {total_params:,}  ({total_params/1e9:.2f}B)")
print(f"Trainable parameters: {trainable_params:,}")
print()

# Print the high-level layer structure
print("Model architecture (top level):")
for name, module in model.named_children():
    print(f"  {name}: {type(module).__name__}")

print()
print("A transformer is a stack of identical 'decoder layers'.")
print("Each layer contains: self-attention + feed-forward network + layer normalisation.")

In [ ]:
# Zoom into one transformer layer to see its components

print("Components inside a single transformer layer (layer 0):")
for name, module in model.model.layers[0].named_modules():
    if name:  # skip the root
        print(f"  {name}: {type(module).__name__}")

### What we're seeing

- `self_attn` — the **self-attention** mechanism. Contains projection matrices `q_proj`, `k_proj`, `v_proj`, `o_proj`. This is where the model learns which tokens to attend to.
- `mlp` — the **feed-forward network**. Applies non-linear transformations to each token's representation.
- `input_layernorm` / `post_attention_layernorm` — **layer normalisation**. Stabilises training.

When we apply LoRA in Notebook 04, we'll insert small trainable matrices alongside the `q_proj` and `v_proj` layers — without touching the original weights.

---

## 4. Running inference — the model's baseline knowledge

In [ ]:
# Let's ask the base model a financial question.
# This shows us the starting point BEFORE any adaptation.
# We'll compare this to the fine-tuned model's answers in Notebook 06.

def generate_response(prompt, max_new_tokens=200, temperature=0.7):
    """
    Run inference on the model.
    
    - prompt: the input text
    - max_new_tokens: maximum tokens to generate
    - temperature: controls randomness. Lower = more deterministic, higher = more creative.
    """
    # Format as a chat message (this model expects a specific template)
    messages = [
        {"role": "system", "content": "You are a helpful financial analyst."},
        {"role": "user", "content": prompt},
    ]
    
    # Apply the chat template — converts the messages dict into the format the model expects
    formatted = tokeniser.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenise and move to device
    inputs = tokeniser(formatted, return_tensors="pt").to(model.device)
    
    # Generate tokens
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokeniser.eos_token_id,
        )
    
    # Decode: strip the input tokens, return only the generated response
    response = tokeniser.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response


# Test 1: General financial question (model should handle this fine)
question = "What is the difference between revenue and profit?"
print(f"Question: {question}")
print("-" * 60)
response = generate_response(question)
print(response)

In [ ]:
# Test 2: A question requiring specific document knowledge
# The model will likely hallucinate or be vague here — it has no access to specific filings.
# This is the problem RAG and fine-tuning solve.

question = "What was Apple's total revenue for fiscal year 2023, and what drove growth?"
print(f"Question: {question}")
print("-" * 60)
print("Base model response (no retrieval, no fine-tuning):")
response = generate_response(question)
print(response)
print()
print("Note: the model may give plausible but vague or incorrect figures.")
print("This is exactly the hallucination problem we address in later notebooks.")

---

## 5. Understanding memory requirements

In [ ]:
# Understanding how much memory models use is important for choosing what to run.
# A rough rule: each parameter in float16 (half precision) uses 2 bytes.

def estimate_model_memory(num_params_billions, precision="float16"):
    bytes_per_param = {"float32": 4, "float16": 2, "int8": 1, "int4": 0.5}
    memory_gb = num_params_billions * 1e9 * bytes_per_param[precision] / (1024**3)
    return memory_gb

models = {
    "TinyLlama (1.1B)": 1.1,
    "Phi-2 (2.7B)": 2.7,
    "Mistral (7B)": 7,
    "Llama-3 (8B)": 8,
    "Llama-3 (70B)": 70,
}

print(f"{'Model':<25} {'float32':>12} {'float16':>12} {'int4 (quantised)':>18}")
print("-" * 70)
for name, params in models.items():
    f32 = estimate_model_memory(params, "float32")
    f16 = estimate_model_memory(params, "float16")
    i4  = estimate_model_memory(params, "int4")
    print(f"{name:<25} {f32:>10.1f}GB {f16:>10.1f}GB {i4:>16.1f}GB")

print()
print("MacBook Air M4 has 16GB or 32GB of unified memory (shared CPU+GPU).")
print("We target models that fit comfortably in float16 — TinyLlama and Phi-2 are ideal.")

---

## Summary

In this notebook we:
- Loaded an open-source LLM from Hugging Face in two lines of code
- Saw how text is broken into tokens before the model processes it
- Inspected the transformer architecture — attention layers, feed-forward networks
- Ran inference and observed the model's baseline capability and its limitations
- Understood memory requirements for different model sizes

The base model is knowledgeable but knows nothing about specific documents or recent events. The next step is to give it that knowledge — either by building a retrieval system (RAG) or retraining it on domain data (fine-tuning).

---

▶ **Next: [Notebook 02 — Data Preparation](02_data_preparation.ipynb)**